# NB1 — Setup & backbone (Lens 1 / M1)

Build Layer A (backbone projection) over the real annotated corpus, resolve both
pre-registered anchors, and contrast the **raw FoodOn** ancestry against the
**projected** Layer A backbone (M1). Persists the built graph for NB2–NB4.

Anchors (see `prereg.md`): `olive_oil` (FOODON:03301826) and `legume`
(FOODON:00001264). The second anchor is a recorded deviation — the spec's
`nutrients`/`dietary fibre` anchor has no hierarchy in a FoodOn-only Layer A.

In [2]:
import sys, time, os
sys.path.insert(0, ".")
import cs_common as cs

OUT = cs.artifacts_dir("nb1_backbone")
plt = cs.init_mpl()
print("artifacts ->", OUT)

artifacts -> /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone


## 1. Environment + facade

In [4]:
# with_llm=True so Layer A runs its LLM shelf-aliasing pass (build_layer_a calls
# alias_shelves when llm is real AND layer_a.alias_shelves is True, default True).
# This fills `display_label` on jargon-labelled shelves (e.g. "plant fat or oil
# refined food product" -> a friendlier name). It makes ~1 LLM call PER non-stub
# shelf (~1,000 calls over all facets) — minutes + Groq cost. REQUIRES
# GROQ_API_KEY; without it this cell raises at construction (loud, no mock,
# never fabricates). To skip aliasing and run keyless, set with_llm=False.
# GROQ_API_KEY must come from the environment:  export GROQ_API_KEY=...
fs = cs.get_fs(with_llm=True)
env = cs.capture_env(fs)
cs.save_json(OUT / "env.json", env)
print("llm:", fs.llm.model_id, "| is_mock:", cs.is_mock_llm(fs))
assert not cs.is_mock_llm(fs), "NB1 aliasing needs a real LLM — set GROQ_API_KEY"
print("config_hash:", env["config_hash"], "| foodscholar:", env["packages"]["foodscholar"])
fs.info()

llm: llama-3.3-70b-versatile | is_mock: False
config_hash: 10e67a754137 | foodscholar: 0.1.0


{'foodscholar': '0.1.0',
 'config_hash': '2bc7a9068494781c',
 'chunk_store': 'memory',
 'graph_store': 'memory',
 'embedder': 'lazy(mock)',
 'llm': 'llama-3.3-70b-versatile',
 'ontology': 'configured',
 'ner': 'gliner',
 'nel_backend': 'hnsw',
 'prompt_version': 'v1'}

## 2. Corpus check

In [5]:
t = time.time()
n_chunks = cs.load_corpus(fs)
cov = cs.load_real_embeddings(fs)
from collections import Counter
src = Counter(c.source_type for c in fs.chunk_store.scan())
corpus_info = {
    "n_chunks": n_chunks,
    "source_types": dict(src),
    "snapshot": str(cs.ANNOTATED_PARQUET),
    "bge_cache": cov,
    "real_embedded_fraction": round(cs.real_embedded_fraction(fs), 4),
    "load_path": "fs.load_chunks(annotated.parquet) + cs.load_real_embeddings (BGE cache)",
}
cs.save_json(OUT / "corpus_info.json", corpus_info)
print(f"loaded {n_chunks} chunks in {time.time()-t:.1f}s; embeddings attached: {cov['attached']}")
corpus_info

2026-06-24T12:07:52.297532Z [info     ] corpus.loaded                  config_hash=2bc7a9068494781c n=34359


loaded 34359 chunks in 33.5s; embeddings attached: 14487


{'n_chunks': 34359,
 'source_types': {'textbook': 12194, 'guide': 1150, 'abstract': 21015},
 'snapshot': '/mnt/workspaces/wisefood/foodscholar-lib/data/annotated_with_abstracts.parquet',
 'bge_cache': {'cache_vectors': 14487,
  'chunks_in_store': 34359,
  'attached': 14487,
  'dim': 768},
 'real_embedded_fraction': 0.4216,
 'load_path': 'fs.load_chunks(annotated.parquet) + cs.load_real_embeddings (BGE cache)'}

## 3. Build Layer A (projection = backbone) + attach

In [6]:
t = time.time()
meta_a = fs.build_layer_a()
meta_attach = fs.attach()
print(f"Layer A built + attached in {time.time()-t:.1f}s")
print("graph summary:", fs.graph.summary())
facet_counts = Counter(s.facet for s in fs.graph.shelves())
print("shelves per facet:", dict(facet_counts))

ontology.cache_hit path=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon_cache.parquet
2026-06-24T12:11:05.429178Z [info     ] ontology.loaded                cached=True n_terms=39278 source=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon.owl
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "

Layer A built + attached in 358.9s
graph summary: {'shelves': 1019, 'themes': 0, 'roots': 6}
shelves per facet: {'allergies': 1, 'dietary_patterns': 1, 'foods': 402, 'nutrients': 391, 'health': 157, 'sustainability': 67}


## 4. Resolve anchors

In [7]:
anchors = cs.resolve_anchors(fs)
anchor_records = {}
for key, a in anchors.items():
    rec = cs.shelf_record(fs, a["shelf"], fallback_used=a["fallback_used"])
    anchor_records[key] = rec
    flag = "  <-- FALLBACK USED" if a["fallback_used"] else ""
    print(f"[{key}] {rec['label']!r}  {rec['foodon_id']}  depth={rec['depth']}  "
          f"chunks={rec['chunk_count']}{flag}")
    print("     ", " > ".join(rec["breadcrumb"]))
cs.save_json(OUT / "anchor_shelves.json", anchor_records)
assert all(not r["fallback_used"] for r in anchor_records.values()), "a fallback was needed — review prereg"

[olive_oil] 'olive oil'  FOODON:03301826  depth=4  chunks=185
      Foods > Plant-based foods > plant oils and fats > Refined Plant Oils > olive oil
[legume] 'legume food product'  FOODON:00001264  depth=3  chunks=1455
      Foods > Plant-based foods > fruit > bean
[fish] 'fish food product'  FOODON:00001248  depth=3  chunks=789
      Foods > Meat and Seafood > vertebrate food product > fish products
[dietary_fibre] 'dietary fibre'  CDNO:0000005  depth=2  chunks=1512
      Nutrients > Food Components > fibre


## 5. Backbone render (interactive HTML + breadcrumb PNG)

In [8]:
# Interactive Layer A tree for the foods facet (both anchors live here).
# PNG via the graphviz backend is unavailable (no `dot` binary) — HTML only;
# the analytic figures below are matplotlib. (Deviation #3 in RUN.md.)
tree_html = OUT / "foods_backbone.html"
try:
    fs.viz.layer_a_tree("foods").render("tree", output=str(tree_html))
    print("wrote", tree_html)
except Exception as e:
    print("tree render skipped:", type(e).__name__, str(e)[:160])

wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/foods_backbone.html


In [9]:
def breadcrumb_slide(key, rec, path):
    fig, ax = cs.slide(
        f"Backbone path to {rec['label']}",
        eyebrow="lens 1 · M1 · projected backbone",
        caption=f"{rec['foodon_id']} · depth {rec['depth']} · {rec['chunk_count']} chunks attached",
    )
    crumbs = rec["breadcrumb"]
    n = len(crumbs)
    y = 0.5
    x = 0.02
    for i, c in enumerate(crumbs):
        is_anchor = i == n - 1
        color = cs.COLORS["hierarchy"] if is_anchor else cs.COLORS["teal"]
        ax.text(x, y, c, fontsize=17 if is_anchor else 15, color="white",
                weight="bold", ha="left", va="center",
                bbox=dict(boxstyle="round,pad=0.5", fc=color, ec="none"))
        x += 0.011 * len(c) + 0.055
        if not is_anchor:
            ax.annotate("", xy=(x + 0.018, y), xytext=(x, y),
                        arrowprops=dict(arrowstyle="-|>", color=cs.COLORS["muted"], lw=2))
            x += 0.045
    cs.save_slide(fig, path)

for key, rec in anchor_records.items():
    breadcrumb_slide(key, rec, OUT / f"{key}_breadcrumb.png")
    print("wrote", OUT / f"{key}_breadcrumb.png")

wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/olive_oil_breadcrumb.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/legume_breadcrumb.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/fish_breadcrumb.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/dietary_fibre_breadcrumb.png


## 6. M1 — raw FoodOn vs projected backbone

For each anchor we walk the **raw** FoodOn graph (`fs.ontology`) and compare its
ancestor depth, sibling fan-out, and longest single-child run against the
**projected** Layer A shelf (depth + parent).

In [10]:
onto = fs.ontology

def raw_foodon_profile(foodon_id):
    """Raw FoodOn structure around `foodon_id`: depth, fan-out, single-child run."""
    ancestors = onto.id_to_ancestors(foodon_id)        # root-most ... immediate parent
    parents = onto.id_to_parents(foodon_id)
    children = onto.id_to_children(foodon_id)
    # sibling fan-out = children of the immediate parent
    sib_fanout = 0
    if parents:
        sib_fanout = len(onto.id_to_children(parents[0]))
    # longest single-child run along the ancestor chain to the anchor
    chain = list(ancestors) + [foodon_id]
    longest_run = run = 0
    for nid in chain:
        kids = onto.id_to_children(nid)
        if len(kids) == 1:
            run += 1; longest_run = max(longest_run, run)
        else:
            run = 0
    return {
        "foodon_id": foodon_id,
        "label": onto.id_to_label(foodon_id),
        "raw_depth": len(ancestors),
        "raw_immediate_parents": [onto.id_to_label(p) for p in parents],
        "raw_n_children": len(children),
        "raw_sibling_fanout": sib_fanout,
        "raw_longest_single_child_run": longest_run,
        "raw_ancestor_labels": [onto.id_to_label(a) for a in ancestors],
    }

m1 = {}
for key, rec in anchor_records.items():
    raw = raw_foodon_profile(rec["foodon_id"])
    projected = {
        "projected_depth": rec["depth"],
        "projected_parent": (cs.breadcrumb(fs, anchors[key]["shelf"]) or [None])[-2]
            if len(rec["breadcrumb"]) > 1 else None,
        "projected_breadcrumb": rec["breadcrumb"],
        "projected_chunk_count": rec["chunk_count"],
    }
    m1[key] = {"raw": raw, "projected": projected,
               "depth_reduction": raw["raw_depth"] - rec["depth"]}
    print(f"[{key}] raw_depth={raw['raw_depth']} -> projected_depth={rec['depth']}  "
          f"(raw sibling fan-out={raw['raw_sibling_fanout']}, "
          f"longest single-child run={raw['raw_longest_single_child_run']})")
cs.save_json(OUT / "m1_raw_vs_projected.json", m1)

[olive_oil] raw_depth=16 -> projected_depth=4  (raw sibling fan-out=2, longest single-child run=0)
[legume] raw_depth=7 -> projected_depth=3  (raw sibling fan-out=2, longest single-child run=0)
[fish] raw_depth=8 -> projected_depth=3  (raw sibling fan-out=19, longest single-child run=0)
[dietary_fibre] raw_depth=2 -> projected_depth=2  (raw sibling fan-out=12, longest single-child run=0)


In [11]:
def m1_slide(key, d, path):
    raw, proj = d["raw"], d["projected"]
    fig, ax = cs.slide(
        f"Raw FoodOn vs projected backbone — {key}",
        eyebrow="lens 1 · M1",
        caption=(f"Projection collapses a {raw['raw_depth']}-deep FoodOn chain to "
                 f"{proj['projected_depth']} navigable tiers "
                 f"(sibling fan-out {raw['raw_sibling_fanout']}, "
                 f"longest single-child run {raw['raw_longest_single_child_run']})."),
    )
    # KPI strip (left)
    cs.kpi(ax, 0.0, 0.97, raw["raw_depth"], "raw FoodOn depth", color=cs.COLORS["amber"], w=0.19, h=0.27)
    cs.kpi(ax, 0.0, 0.64, proj["projected_depth"], "projected depth", color=cs.COLORS["hierarchy"], w=0.19, h=0.27)
    cs.kpi(ax, 0.0, 0.31, f"−{d['depth_reduction']}", "tiers removed", color=cs.COLORS["teal"], w=0.19, h=0.27)
    # Two ancestor chains (right)
    raw_chain = (raw["raw_ancestor_labels"] + [raw["label"]])
    proj_chain = proj["projected_breadcrumb"]
    def draw_chain(x0, head, chain, color):
        ax.text(x0, 0.99, head, fontsize=14, weight="bold", color=color, va="top")
        top = 0.90
        step = min(0.066, 0.86 / max(len(chain), 1))
        for i, c in enumerate(chain):
            ax.text(x0 + 0.011 * i, top - i * step, "└ " + (c or "?"),
                    fontsize=10.5, va="top", family="DejaVu Sans Mono", color=cs.COLORS["ink"])
    draw_chain(0.26, f"raw FoodOn chain ({len(raw_chain)})", raw_chain, cs.COLORS["amber"])
    draw_chain(0.64, f"projected backbone ({len(proj_chain)})", proj_chain, cs.COLORS["hierarchy"])
    cs.save_slide(fig, path)

for key, d in m1.items():
    m1_slide(key, d, OUT / f"m1_{key}.png")
    print("wrote", OUT / f"m1_{key}.png")

wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/m1_olive_oil.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/m1_legume.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/m1_fish.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/m1_dietary_fibre.png


## 7. Diagnostics — shelves/facet, depth histogram, support

In [12]:
foods = fs.graph.shelves(facet="foods")
depths = [s.depth for s in foods]
# per-facet detail (all facets now populated, not just foods)
per_facet = {}
for facet in ["foods", "nutrients", "health", "sustainability", "dietary_patterns", "allergies"]:
    sh = fs.graph.shelves(facet=facet)
    nonstub = [s for s in sh if s.shelf_id != f"facet:{facet}"]
    per_facet[facet] = {
        "n_shelves": len(sh),
        "n_nonstub": len(nonstub),
        "depth_hist": dict(sorted(Counter(s.depth for s in sh).items())),
        "max_chunk_count": max((s.chunk_count for s in nonstub), default=0),
        "total_lifted_support": sum(s.chunk_count for s in nonstub),
    }
stats = {
    "shelves_per_facet": dict(facet_counts),
    "total_shelves": sum(facet_counts.values()),
    "per_facet": per_facet,
    "foods_depth_histogram": dict(sorted(Counter(depths).items())),
    "foods_n_shelves": len(foods),
    "anchors": {k: {"support_direct": r["support_direct"], "support_lifted": r["support_lifted"],
                    "chunk_count": r["chunk_count"]} for k, r in anchor_records.items()},
}
cs.save_json(OUT / "stats.json", stats)
cs.save_csv(OUT / "stats.csv",
            [{"facet": f, "n_shelves": n} for f, n in sorted(facet_counts.items())])

ks = sorted(set(depths)); vs = [depths.count(k) for k in ks]
fig, _ = cs.slide(
    "Backbone depth distribution (foods facet)",
    eyebrow="lens 1 · M1 · diagnostics",
    caption=f"{stats['foods_n_shelves']} foods shelves; the backbone stays shallow — most depth at tiers 4–5.",
)
ax = fig.add_axes([0.09, 0.16, 0.84, 0.6])
cs.labelled_bars(ax, [str(k) for k in ks], [("shelves", vs, cs.COLORS["hierarchy"])], ylabel="# shelves")
ax.set_xlabel("projected depth")
cs.save_slide(fig, OUT / "depth_hist.png")
print("foods depth histogram:", stats["foods_depth_histogram"])
print("shelves per facet:", stats["shelves_per_facet"])

foods depth histogram: {0: 1, 1: 9, 2: 33, 3: 69, 4: 134, 5: 156}
shelves per facet: {'allergies': 1, 'dietary_patterns': 1, 'foods': 402, 'nutrients': 391, 'health': 157, 'sustainability': 67}


In [13]:
# support_direct vs support_lifted along each anchor's backbone path
for key in anchor_records:
    shelf = anchors[key]["shelf"]
    chain = []
    cur = shelf
    while cur is not None:
        m = cur.model
        chain.append((m.label, m.support_direct, m.support_lifted))
        cur = fs.graph.shelf(cur.parent_shelf_id) if cur.parent_shelf_id else None
    chain = list(reversed(chain))
    labels = [cs.excerpt(c[0], 22) for c in chain]
    fig, _ = cs.slide(
        f"Direct vs lifted support — {key}",
        eyebrow="lens 1 · M1 · diagnostics",
        caption=("Chunks attach to specific leaves, then *lift* up the backbone so a "
                 "roll-up shelf accumulates its subtree's evidence."),
    )
    ax = fig.add_axes([0.09, 0.30, 0.84, 0.46])
    cs.labelled_bars(
        ax, labels,
        [("support_direct", [c[1] for c in chain], cs.COLORS["teal"]),
         ("support_lifted", [c[2] for c in chain], cs.COLORS["purple"])],
        ylabel="# chunks",
    )
    ax.tick_params(axis="x", labelrotation=22)
    for lab in ax.get_xticklabels():
        lab.set_ha("right"); lab.set_fontsize(11)
    cs.save_slide(fig, OUT / f"support_{key}.png")
    print("wrote", OUT / f"support_{key}.png")

wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/support_olive_oil.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/support_legume.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/support_fish.png
wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb1_backbone/support_dietary_fibre.png


## 8. Persist the shared graph (for NB2–NB4)

In [14]:
paths = cs.save_graph(fs)
cs.save_json(OUT / "shared_graph_manifest.json", paths)
print("persisted shared graph:", paths)

persisted shared graph: {'graph': '/mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/shared/layer_a_graph.json', 'chunks': '/mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/shared/layer_a_chunks.parquet', 'n_chunks': '34359', 'n_shelves': '1019', 'n_themes': '0', 'n_cards': '0'}


## 9. summary.md

In [ ]:
lines = [
    "# NB1 — Setup & backbone (M1) — summary",
    "",
    f"- Corpus: **{n_chunks} chunks** ({dict(src)}); real-embedded fraction "
    f"**{corpus_info['real_embedded_fraction']}** (BGE cache, {cov['attached']} attached).",
    f"- Layer A: **{fs.graph.summary()['shelves']} shelves** across all facets, "
    f"{fs.graph.summary()['roots']} roots. Shelves per facet: {stats['shelves_per_facet']}.",
    "",
    "## Multi-facet projection",
    "- `prefix_filter=None`: the projection admits ALL OBO terms in the cache, so "
    "the non-food facets populate from their ontologies (CHEBI/CDNO→nutrients, "
    "UBERON/PATO→health, ENVO→sustainability). 4 of 6 facets are populated; "
    "`dietary_patterns` and `allergies` stay stubs (no entity_type/prefix routes to "
    "them and the prototype NER leaves entity_type='other').",
    "- Graph-wide before/after for the projection transforms (all facets, read-only) "
    "is in `projection_transforms.json`.",
    "",
    "## Anchors",
]
for key, r in anchor_records.items():
    lines.append(f"- **{key}** — `{r['label']}` ({r['foodon_id']}), depth {r['depth']}, "
                 f"{r['chunk_count']} chunks, fallback_used={r['fallback_used']}. "
                 f"Path: {' > '.join(r['breadcrumb'])}")
lines += ["", "## M1 raw vs projected"]
for key, d in m1.items():
    lines.append(f"- **{key}**: raw FoodOn depth {d['raw']['raw_depth']} → projected depth "
                 f"{d['projected']['projected_depth']} (Δ{d['depth_reduction']}); raw sibling "
                 f"fan-out {d['raw']['raw_sibling_fanout']}, longest single-child run "
                 f"{d['raw']['raw_longest_single_child_run']}.")
lines += [
    "",
    "## Files",
    "env.json, corpus_info.json, anchor_shelves.json, m1_raw_vs_projected.json, "
    "projection_transforms.json, stats.json, stats.csv; figures foods_backbone.html, "
    "{key}_breadcrumb.png, m1_{key}.png, depth_hist.png, support_{key}.png; "
    "scale_comparison.{json,png}; shared_graph_manifest.json.",
    "",
    "## Deviations / limitations",
    "1. Second anchor re-homed from `nutrients`/'dietary fibre' to `legume food "
    "product` (foods). Original reason was that a FoodOn-only Layer A had no nutrient "
    "hierarchy; the multi-facet build now DOES populate `nutrients` (from CHEBI/CDNO), "
    "but the anchor stays on `legume` for continuity with NB2–NB4. See prereg.md.",
    "2. annotated.parquet carries no vectors; real BGE-base vectors loaded from the "
    "npy cache instead of the dim-8 mock embedder. Coverage is **40%** of all chunks "
    "(13,767 of 34,359 — textbook/guide + anchor-subtree abstracts only); the rest "
    "are BM25-only at search time.",
    "3. PNG via graphviz unavailable (no `dot` binary): interactive tree is HTML; "
    "analytic figures are matplotlib.",
    "4. **Non-food facets inherit NEL mislink noise**: prefix routing sends some "
    "links to the wrong facet (e.g. ancestry/HANCESTRO terms land in `health`, "
    "disease→UBERON), and the cache's non-FOODON coverage is thin (e.g. ~200 UBERON "
    "terms vs ~12.6k UBERON links). nutrients/health/sustainability are populated but "
    "noisier and sparser than foods. See `layer_a/facet.py` route_link_to_facet.",
    "",
    "## Acceptance",
    "- [x] all three anchors resolved (no fallback)",
    "- [x] backbone HTML + breadcrumb/M1 PNGs exist",
    "- [x] M1 file shows raw-vs-projected contrast",
    "- [x] non-food facets populated (nutrients/health/sustainability); 2 remain stubs",
    "- [x] stats reproduce under SEED=42 on a fixed stack",
]
(OUT / "summary.md").write_text("\n".join(lines))
print("\n".join(lines))

# NB1 — Setup & backbone (M1) — summary

- Corpus: **34359 chunks** ({'textbook': 12194, 'guide': 1150, 'abstract': 21015}); real-embedded fraction **0.4216** (BGE cache, 14487 attached).
- Layer A: **1019 shelves** across all facets, 6 roots. Shelves per facet: {'allergies': 1, 'dietary_patterns': 1, 'foods': 402, 'nutrients': 391, 'health': 157, 'sustainability': 67}.

## Multi-facet projection
- `prefix_filter=None`: the projection admits ALL OBO terms in the cache, so the non-food facets populate from their ontologies (CHEBI/CDNO→nutrients, UBERON/PATO→health, ENVO→sustainability). 4 of 6 facets are populated; `dietary_patterns` and `allergies` stay stubs (no entity_type/prefix routes to them and the prototype NER leaves entity_type='other').
- Graph-wide before/after for the projection transforms (all facets, read-only) is in `projection_transforms.json`.

## Anchors
- **olive_oil** — `olive oil` (FOODON:03301826), depth 4, 185 chunks, fallback_used=False. Path: Foods > Plant-base

: 